# PRÁTICA 4 — Encoder — BERTimbau

**Grupo 6** · STIL 2017 · corpus herdado da PRÁTICA 1

`corpus ▷ ajuste do BERTimbau ▷ vetores ▷ similaridade ▷ proposta de classificação de estilo`

## Atividades

**1) Vetores gerados para as palavras:**

- a) modelos
- b) linguagem
- c) substantivo de maior frequência do corpus
- d) verbo de maior frequência do corpus

**2) Quais termos são mais similares aos termos:**

- a) modelos
- b) linguagem
- c) substantivo de maior frequência do corpus
- d) verbo de maior frequência do corpus

**3) Propor uma tarefa de classificação de estilo de escrita:**

- Estilo acadêmico
- Estilo narrativo
- Estilo descritivo

O notebook conduz as etapas e registra as escolhas; `ml/` reúne os detalhes de implementação.


## 0. Setup

- **Local:** `uv sync` dentro de `pratica-4-bert/` cria o ambiente com o kernel.
- **Colab:** a célula abaixo clona o repositório e entra na pasta da prática. `REF` permite selecionar uma branch publicada.
- O snapshot acompanha o projeto; não é necessário baixar os PDFs novamente.
- No Colab, selecione GPU para o treinamento. Localmente, usamos CUDA, MPS ou CPU.
- O setup instala as dependências no Colab; o modelo é baixado na primeira execução.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/joaoleahy/topicos-banco-de-dados-ii.git"
REF = None  # None = branch padrão; ou informe a branch/tag publicada.
PASTA = "pratica-4-bert"


def ensure_project() -> Path:
    candidatos = [Path.cwd(), Path.cwd() / PASTA]
    if IN_COLAB:
        dest = Path("/content/topicos-banco-de-dados-ii")
        if not any((p / "ml/corpus.py").is_file() for p in candidatos):
            if not dest.exists():
                comando = ["git", "clone", "--depth", "1"]
                if REF:
                    comando += ["--branch", REF]
                subprocess.check_call(comando + [REPO, str(dest)])
            candidatos.append(dest / PASTA)

    for candidato in candidatos:
        if (candidato / "ml/corpus.py").is_file():
            raiz = candidato.resolve()
            if IN_COLAB and not (raiz / "corpus/stil2017-pt.jsonl").is_file():
                raise FileNotFoundError("O checkout não contém o snapshot da Prática 4. Publique também corpus/stil2017-pt.jsonl.")
            os.chdir(raiz)
            return raiz
    raise FileNotFoundError(
        f"Não encontrei {PASTA}/ml/corpus.py. Localmente, abra na raiz do repositório "
        "ou na pasta da prática. No Colab, confira se a Prática 4 foi publicada em REPO/REF. "
        "Se o clone já existia, ele é reutilizado sem atualização automática."
    )


ROOT = ensure_project()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)
print("Colab:", IN_COLAB)

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch>=2.6", "transformers>=4.57,<5"])


Projeto: /Users/antoniel/d/ufba/bd-2-llms/topicos-banco-de-dados-ii/pratica-4-bert
Colab: False


## 1. Corpus — artigos do STIL 2017

O corpus vem dos artigos do **STIL 2017**, processados na PRÁTICA 1.

- `artigo_completo`: corpo do artigo com pontuação, acentos e maiúsculas preservados; sem autores e referências.
- `pos_tagger`: classe gramatical de cada palavra, usada na contagem dos substantivos e verbos.

Mantemos as stopwords e o texto dos artigos para a entrada do modelo.


In [2]:
from ml.corpus import carregar, frequencias, palavras_alvo

artigos = carregar()
print(f"Artigos em português: {len(artigos)}")
print(f"Caracteres do corpus: {sum(len(a['artigo_completo']) for a in artigos):,}")


Artigos em português: 22
Caracteres do corpus: 348,817


In [3]:
primeiro = artigos[0]
print(primeiro['id'], '—', primeiro['titulo'])
print(primeiro['artigo_completo'][:1000])


article_01 — Uma Ferramenta para Identificar Desvios de Linguagem na Língua Portuguesa
Os desvios de linguagem são palavras ou construções que ferem as normas gramaticais e costumam ocorrer por descuido ou desconhecimento das regras da língua. De acordo com, o público-alvo dos revisores gramaticais tem se revelado insatisfeito com a restrição de intervenção aos problemas ortográfico-sintáticos, exigindo a consideração de problemas relacionados ao estilo, muito mais vinculados à eficácia comunicativa do que a simples adequação à norma gramatical.
Nesse contexto, este trabalho apresenta uma ferramenta para identificação de desvios de linguagem em textos descritivos formais. Uma ferramenta com tal finalidade pode ter várias aplicações, tanto no âmbito acadêmico quanto empresarial, desde sistemas de correção automática de redações até revisão de artigos e documentos. Em pesquisas na literatura, identificamos a ferramenta LanguageTool que detecta apenas alguns tipos de desvios considerados 

## 2. Palavras-alvo — substantivo e verbo mais frequentes

A classe gramatical vem do `pos_tagger` da PRÁTICA 1 (spaCy). Contamos pela **forma em minúsculas**, sem lematizar, como na PRÁTICA 3.

Usamos `NOUN` para substantivos e `VERB` para verbos; auxiliares (`AUX`) ficam fora da contagem.


In [4]:
for classe in ("NOUN", "VERB"):
    print(f"\n{classe} — dez formas mais frequentes")
    for palavra, quantidade in frequencias(artigos, classe).most_common(10):
        print(f"{palavra:20} {quantidade}")

print("\nPalavras-alvo da atividade:")
for item, palavra in palavras_alvo(artigos).items():
    print(f"{item}: {palavra}")



NOUN — dez formas mais frequentes
palavras             156
dados                154
base                 153
termos               139
construções          124
construção           116
exemplo              114
trabalho             109
texto                109
corpus               105

VERB — dez formas mais frequentes
pode                 79
podem                62
partir               52
apresenta            49
tem                  40
há                   40
seja                 39
identificar          38
obtidos              33
utilizado            31

Palavras-alvo da atividade:
a) palavra indicada: modelos
b) palavra indicada: linguagem
c) substantivo mais frequente: palavras
d) verbo mais frequente: pode


## 3. Treino e validação

Separamos os artigos com **seed 42**: 80% para treino e 20% para validação, arredondando a quantidade de validação para cima. A divisão ocorre antes da tokenização; um artigo pertence a apenas um conjunto.


In [5]:
from ml.preparar import separar

SEED = 42
FRACAO_VALIDACAO = 0.20

# Dividir por artigo impede que trechos do mesmo texto apareçam nos dois conjuntos.
artigos_treino, artigos_validacao = separar(
    artigos, fracao_validacao=FRACAO_VALIDACAO, seed=SEED,
)
print("Treino:", len(artigos_treino), "artigos")
print("Validação:", len(artigos_validacao), "artigos")
print("IDs de validação:", [a["id"] for a in artigos_validacao])


Treino: 17 artigos
Validação: 5 artigos
IDs de validação: ['article_31', 'article_11', 'article_03', 'article_24', 'article_17']


## 4. Tokenização — BERTimbau Base

Usamos o tokenizador de `neuralmind/bert-base-portuguese-cased`, preservando maiúsculas e minúsculas.

Blocos de **128 tokens**, incluindo `[CLS]` e `[SEP]`. Cada bloco pertence a um artigo; o último bloco recebe padding, sem descartar o restante do texto.


In [6]:
from transformers import AutoTokenizer
from ml.preparar import tokenizar_artigos, criar_blocos

MODELO = "neuralmind/bert-base-portuguese-cased"
TAMANHO_BLOCO = 128

tokenizer = AutoTokenizer.from_pretrained(MODELO)

# Tokenizar preserva cada artigo; criar blocos limita o tamanho de entrada do BERT.
tokens_treino = tokenizar_artigos(artigos_treino, tokenizer)
tokens_validacao = tokenizar_artigos(artigos_validacao, tokenizer)

blocos_treino = criar_blocos(tokens_treino, tokenizer, tamanho=TAMANHO_BLOCO)
blocos_validacao = criar_blocos(tokens_validacao, tokenizer, tamanho=TAMANHO_BLOCO)

print("Blocos de treino:", len(blocos_treino))
print("Blocos de validação:", len(blocos_validacao))
print("Exemplo:", tokenizer.convert_ids_to_tokens(blocos_treino[0]["input_ids"][:30]))


/Users/antoniel/d/ufba/bd-2-llms/topicos-banco-de-dados-ii/pratica-4-bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Blocos de treino: 516
Blocos de validação: 147
Exemplo: ['[CLS]', 'O', 'presente', 'trabalho', 'descreve', 'a', 'construção', 'de', 'uma', 'mini', '##gram', '##ática', 'computacional', 'para', 'o', 'português', 'brasileiro', '(', 'dor', '##ava', '##nte', 'P', '##B', ')', '.', 'Esse', 'frag', '##mento', 'de', 'gramática']


## 5. Ajuste — palavras mascaradas (MLM)

| Parâmetro | Valor |
|---|---|
| Modelo | BERTimbau Base |
| Limite de épocas | 15 |
| Paciência do early stopping | 3 épocas sem melhora |
| Lote | 8 blocos |
| Taxa de aprendizado | 0,00002 |
| Tokens selecionados para MLM | 15% |
| Seed | 42 |

Dos tokens selecionados, 80% são substituídos por `[MASK]`, 10% por um token aleatório e 10% são mantidos. Padding e tokens especiais não entram como alvos.

As máscaras variam no treino e permanecem fixas na validação. Registramos a perda antes do ajuste e ao final de cada época; salvamos a melhor época ajustada em `modelos/bertimbau-stil/`.


### 5.1. Máscaras e lotes

Máscaras novas a cada passagem pelo treino; máscaras fixas para comparar as avaliações nas mesmas condições.


In [7]:
from ml.treinar import (
    fixar_semente, escolher_dispositivo, criar_mascarador, criar_lotes,
    fixar_mascaras, carregar_modelo, criar_otimizador, avaliar,
    treinar_epoca, salvar_modelo, salvar_resultado,
)

PROBABILIDADE_MASCARA = 0.15
TAMANHO_LOTE = 8

fixar_semente(SEED)
dispositivo = escolher_dispositivo()
mascarador = criar_mascarador(tokenizer, probabilidade=PROBABILIDADE_MASCARA)

lotes_validacao = criar_lotes(
    blocos_validacao, mascarador, tamanho_lote=TAMANHO_LOTE,
    embaralhar=False, seed=SEED,
)
validacao_fixa = fixar_mascaras(lotes_validacao)

lotes_treino = criar_lotes(
    blocos_treino, mascarador, tamanho_lote=TAMANHO_LOTE,
    embaralhar=True, seed=SEED,
)
print("Dispositivo:", dispositivo)


Dispositivo: mps


### 5.2. Modelo inicial e avaliação de referência

Carregamos os pesos pré-treinados e medimos a perda antes de atualizar o modelo.


In [8]:
TAXA_APRENDIZADO = 2e-5
WEIGHT_DECAY = 0.01

modelo = carregar_modelo(MODELO, dispositivo)
otimizador = criar_otimizador(
    modelo, taxa_aprendizado=TAXA_APRENDIZADO, weight_decay=WEIGHT_DECAY,
)

perda_antes = avaliar(modelo, validacao_fixa, dispositivo)
print(f"Validação antes do ajuste: {perda_antes:.4f}")


Some weights of the model checkpoint at neuralmind/bert-base-portuguese-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Validação antes do ajuste: 2.4870


### 5.3. Treinar, avaliar e selecionar

Após cada época, avaliamos o modelo nos mesmos alvos de validação. Salvamos a época ajustada com menor perda.

Treinamos por até **15 épocas**. Qualquer redução da perda de validação conta como melhora e zera a contagem de espera. Paramos após **3 épocas consecutivas sem melhora** e recarregamos o melhor modelo salvo.


In [9]:
MAX_EPOCAS = 15
PACIENCIA = 3
LIMITE_GRADIENTE = 1.0
DESTINO = ROOT / "modelos/bertimbau-stil"

historico = []
melhor_perda = float("inf")
melhor_epoca = None
epocas_sem_melhora = 0
motivo_parada = "limite de épocas"

for epoca in range(1, MAX_EPOCAS + 1):
    print(f"Época {epoca}/{MAX_EPOCAS}")
    perda_treino = treinar_epoca(
        modelo, lotes_treino, otimizador, dispositivo,
        limite_gradiente=LIMITE_GRADIENTE,
    )
    perda_validacao = avaliar(modelo, validacao_fixa, dispositivo)

    historico.append({
        "epoca": epoca,
        "loss_treino": perda_treino,
        "loss_validacao": perda_validacao,
    })
    print(f"Treino: {perda_treino:.4f}; validação: {perda_validacao:.4f}")

    # O critério de seleção é a validação, não a perda do treino.
    if perda_validacao < melhor_perda:
        melhor_perda = perda_validacao
        melhor_epoca = epoca
        salvar_modelo(modelo, tokenizer, DESTINO)
        epocas_sem_melhora = 0
    else:
        epocas_sem_melhora += 1
        print(f"Sem melhora: {epocas_sem_melhora}/{PACIENCIA}")

    if epocas_sem_melhora >= PACIENCIA:
        motivo_parada = "early stopping"
        print(f"Parada antecipada na época {epoca}; melhor época: {melhor_epoca}.")
        break

# As próximas etapas devem usar a melhor época, não necessariamente a última.
del otimizador, modelo
modelo = carregar_modelo(DESTINO, dispositivo)


Época 1/15


Lote 20/65


Lote 40/65


Lote 60/65


Treino: 2.2237; validação: 1.8229


Época 2/15


... (26 outputs left out)

Lote 20/65


Lote 40/65


Lote 60/65


Treino: 1.5832; validação: 1.6829
Sem melhora: 3/3
Parada antecipada na época 8; melhor época: 5.


### 5.4. Registro da execução

Salvamos parâmetros, divisão dos artigos e histórico junto ao modelo selecionado.


In [10]:
resultado = {
    "modelo_base": MODELO,
    "seed": SEED,
    "device": dispositivo,
    "fracao_validacao": FRACAO_VALIDACAO,
    "tamanho_bloco": TAMANHO_BLOCO,
    "max_epocas": MAX_EPOCAS,
    "paciencia": PACIENCIA,
    "epocas": len(historico),
    "motivo_parada": motivo_parada,
    "batch_size": TAMANHO_LOTE,
    "learning_rate": TAXA_APRENDIZADO,
    "weight_decay": WEIGHT_DECAY,
    "limite_gradiente": LIMITE_GRADIENTE,
    "mlm_probability": PROBABILIDADE_MASCARA,
    "loss_validacao_antes": perda_antes,
    "melhor_epoca": melhor_epoca,
    "historico": historico,
    "artigos_treino": sorted(a["id"] for a in artigos_treino),
    "artigos_validacao": sorted(a["id"] for a in artigos_validacao),
}
salvar_resultado(resultado, DESTINO / "treinamento.json")


## 6. Resultado do ajuste

A perda de validação é calculada por token-alvo. A melhor época é escolhida pela validação; esse conjunto não constitui um teste independente.


In [11]:
print(f"Perda de validação antes: {resultado['loss_validacao_antes']:.4f}")
for linha in resultado['historico']:
    print(f"Época {linha['epoca']}: treino={linha['loss_treino']:.4f}; validação={linha['loss_validacao']:.4f}")
print("Época salva:", resultado['melhor_epoca'])
print("Modelo salvo em:", ROOT / "modelos/bertimbau-stil")
print("Épocas executadas:", resultado["epocas"])
print("Motivo da parada:", resultado["motivo_parada"])


Perda de validação antes: 2.4870
Época 1: treino=2.2237; validação=1.8229
Época 2: treino=1.9684; validação=1.7596
Época 3: treino=1.8029; validação=1.7329
Época 4: treino=1.8091; validação=1.7012
Época 5: treino=1.6934; validação=1.6828
Época 6: treino=1.6172; validação=1.6885
Época 7: treino=1.6395; validação=1.6908
Época 8: treino=1.5832; validação=1.6829
Época salva: 5
Modelo salvo em: /Users/antoniel/d/ufba/bd-2-llms/topicos-banco-de-dados-ii/pratica-4-bert/modelos/bertimbau-stil
Épocas executadas: 8
Motivo da parada: early stopping


## 7. Vetores — da ocorrência à palavra

`texto ▷ subtokens ▷ última camada do encoder ▷ média por ocorrência ▷ média por palavra`

Usamos o modelo da melhor época e o corpus completo para a análise lexical. Esta etapa é descritiva; não mede generalização em dados inéditos.

- Contextos de até **128 tokens**, sem cortar uma palavra entre blocos.
- Texto sem máscaras e modelo em modo de avaliação.
- Média dos subtokens de cada ocorrência; depois, média das ocorrências da mesma forma em minúsculas.
- Cada vetor tem **768 dimensões**. Singular e plural continuam separados.

Pontuação participa do contexto, mas não recebe vetor de palavra. Formas com hífen ou apóstrofo são agrupadas; ocorrências com `[UNK]` são descartadas. A contagem de ocorrências desta etapa usa o alinhamento com o texto, podendo diferir da anotação POS usada para escolher os alvos.


In [2]:
from ml.vetores import (
    preparar_contextos, extrair_ocorrencias, acumular_ocorrencias,
    calcular_medias, selecionar_candidatos, normalizar_vetores,
    buscar_similares, salvar_vetores, exportar_alvos,
)

TAMANHO_CONTEXTO = 128
LOTE_EXTRACAO = 8

# Reutilizamos o encoder do melhor modelo recarregado após o early stopping.
encoder = modelo.bert
contextos = preparar_contextos(artigos, tokenizer, tamanho=TAMANHO_CONTEXTO)
print("Contextos:", len(contextos))
print("Dimensão do encoder:", encoder.config.hidden_size)


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Contextos: 666
Dimensão do encoder: 768


In [3]:
# Cada ocorrência recebe a média dos seus subtokens na última camada.
ocorrencias = extrair_ocorrencias(
    encoder, contextos, dispositivo, tamanho_lote=LOTE_EXTRACAO,
)
somas, contagens = acumular_ocorrencias(ocorrencias)

# Cada forma recebe a média de suas ocorrências no corpus.
vetores = calcular_medias(somas, contagens)
print("Formas com vetor:", len(vetores))
print("Ocorrências representadas:", sum(contagens.values()))


Formas com vetor: 7146
Ocorrências representadas: 52117


## 8. Atividade 1 — vetores gerados

Mostramos as primeiras 12 dimensões de cada vetor. Os 768 valores completos são exportados em `resultados/palavras-alvo.json`.


In [4]:
alvos = palavras_alvo(artigos)

for item, palavra in alvos.items():
    vetor = vetores[palavra]
    print(f"\n{item}: {palavra}")
    print("Ocorrências representadas:", contagens[palavra])
    print("Dimensões:", len(vetor))
    print("Primeiras 12:", [round(valor, 6) for valor in vetor[:12].tolist()])



a) palavra indicada: modelos
Ocorrências representadas: 14
Dimensões: 768
Primeiras 12: [-0.034729, 0.031901, 1.035421, 0.079058, -0.035782, 0.211485, -0.490587, 0.134293, 0.625734, -1.153615, 0.117164, -0.072391]

b) palavra indicada: linguagem
Ocorrências representadas: 49
Dimensões: 768
Primeiras 12: [-0.068037, -0.104626, 0.77587, -0.291047, 0.49343, -0.269177, -0.603369, -0.002217, -0.41786, 0.229131, 0.105608, -0.438008]

c) substantivo mais frequente: palavras
Ocorrências representadas: 163
Dimensões: 768
Primeiras 12: [-0.345307, 0.021097, 1.204398, -0.300739, 0.231097, -0.387647, -0.27389, 0.405194, -0.225973, 0.073914, -0.135611, 0.672727]

d) verbo mais frequente: pode
Ocorrências representadas: 79
Dimensões: 768
Primeiras 12: [-0.202856, -0.005593, 1.555687, 0.104424, -0.380079, 0.277681, 0.028261, 0.487121, -0.397432, -0.044456, 0.629568, -0.930058]


## 9. Atividade 2 — termos mais similares

Normalizamos os vetores e calculamos a similaridade de cosseno. Buscamos os **10 vizinhos** de cada alvo, excluindo a própria palavra.

Candidatos: formas com pelo menos **3 ocorrências** e **2 caracteres**. Stopwords permanecem elegíveis. A similaridade representa proximidade dos vetores médios; não implica sinonímia. A média reúne os diferentes contextos e sentidos de uma palavra.


In [5]:
FREQUENCIA_MINIMA = 3
TAMANHO_MINIMO = 2
QUANTIDADE_VIZINHOS = 10

candidatos = selecionar_candidatos(
    contagens, frequencia_minima=FREQUENCIA_MINIMA, tamanho_minimo=TAMANHO_MINIMO,
)
normalizados = normalizar_vetores(vetores)

rankings = {}
for palavra in alvos.values():
    rankings[palavra] = buscar_similares(
        palavra, normalizados, candidatos, quantidade=QUANTIDADE_VIZINHOS,
    )
print("Candidatos à busca:", len(candidatos))


Candidatos à busca: 2411


### a) modelos


In [6]:
palavra = alvos['a) palavra indicada']
print("Alvo:", palavra)
for posicao, (termo, cosseno) in enumerate(rankings[palavra], 1):
    print(f"{posicao:2}. {termo:24} {cosseno:.4f}")


Alvo: modelos
 1. modelo                   0.8064
 2. algoritmos               0.6615
 3. modelagem                0.6491
 4. exemplos                 0.6363
 5. sistemas                 0.6302
 6. métodos                  0.6198
 7. cenários                 0.6188
 8. metodos                  0.6046
 9. conceitos                0.6010
10. conjuntos                0.5937


### b) linguagem


In [7]:
palavra = alvos['b) palavra indicada']
print("Alvo:", palavra)
for posicao, (termo, cosseno) in enumerate(rankings[palavra], 1):
    print(f"{posicao:2}. {termo:24} {cosseno:.4f}")


Alvo: linguagem
 1. língua                   0.7976
 2. linguística              0.7285
 3. línguas                  0.7084
 4. linguísticos             0.6862
 5. idioma                   0.6654
 6. linguístico              0.6602
 7. discurso                 0.6556
 8. fala                     0.6536
 9. gramática                0.6533
10. gramaticais              0.6518


### c) substantivo de maior frequência


In [8]:
palavra = alvos['c) substantivo mais frequente']
print("Alvo:", palavra)
for posicao, (termo, cosseno) in enumerate(rankings[palavra], 1):
    print(f"{posicao:2}. {termo:24} {cosseno:.4f}")


Alvo: palavras
 1. palavra                  0.8240
 2. palavras-chave           0.7537
 3. expressões               0.7520
 4. termos                   0.7186
 5. orações                  0.6991
 6. letras                   0.6947
 7. verbais                  0.6796
 8. pronomes                 0.6677
 9. terminações              0.6657
10. ideias                   0.6644


### d) verbo de maior frequência


In [9]:
palavra = alvos['d) verbo mais frequente']
print("Alvo:", palavra)
for posicao, (termo, cosseno) in enumerate(rankings[palavra], 1):
    print(f"{posicao:2}. {termo:24} {cosseno:.4f}")


Alvo: pode
 1. podem                    0.8414
 2. pode-se                  0.7310
 3. poderia                  0.6863
 4. possível                 0.6826
 5. podendo                  0.6804
 6. permite                  0.6543
 7. possibilidade            0.6453
 8. possa                    0.6287
 9. tem                      0.6130
10. podemos                  0.5967


### Leitura dos resultados

- **modelos:** aproxima-se de `modelo`, `algoritmos` e `modelagem`.
- **linguagem:** aproxima-se de `língua`, `linguística` e `línguas`.
- **palavras:** aproxima-se de `palavra`, `palavras-chave` e `expressões`.
- **pode:** aproxima-se de `podem`, `pode-se` e `poderia`.

Os vizinhos incluem flexões da mesma palavra e termos relacionados ao seu uso nos artigos. A extração dos PDFs ainda deixa variantes como `métodos` e `metodos` no vocabulário. Os resultados descrevem o modelo ajustado; não fizemos aqui uma comparação com o BERTimbau antes do ajuste.


## 10. Salvar — vetores e consultas

`vetores.pt` guarda os vetores e contagens do vocabulário. `palavras-alvo.json` contém os quatro itens, vetores completos e vizinhos; `similares.csv` reúne o ranking de similaridade.


In [10]:
PASTA_RESULTADOS = ROOT / "resultados"

salvar_vetores(vetores, contagens, PASTA_RESULTADOS / "vetores.pt")
exportar_alvos(alvos, vetores, contagens, rankings, PASTA_RESULTADOS)
salvar_resultado({
    "modelo": str(DESTINO),
    "epoca": resultado["melhor_epoca"],
    "camada": "última",
    "agregacao": "média de subtokens por ocorrência; média de ocorrências por forma",
    "tamanho_contexto": TAMANHO_CONTEXTO,
    "frequencia_minima": FREQUENCIA_MINIMA,
    "tamanho_minimo": TAMANHO_MINIMO,
    "artigos": [a["id"] for a in artigos],
}, PASTA_RESULTADOS / "extracao.json")
print("Resultados salvos em:", PASTA_RESULTADOS)


Resultados salvos em: /Users/antoniel/d/ufba/bd-2-llms/topicos-banco-de-dados-ii/pratica-4-bert/resultados


## Referências

- [BERTimbau — modelo e tokenizador](https://huggingface.co/neuralmind/bert-base-portuguese-cased)
- [Hugging Face — Masked language modeling](https://huggingface.co/docs/transformers/tasks/masked_language_modeling)
- [Hugging Face — alinhamento do tokenizador](https://huggingface.co/docs/transformers/main_classes/tokenizer)
- [Hugging Face — saídas do encoder BERT](https://huggingface.co/docs/transformers/model_doc/bert)
